# WSJ 2027 - Östergötland

Statistik och gruppindelning för alla deltagare, IST och CMT från Östergötlands kårer.

Data: Scoutnet API + gruppindelnings-CSV:er (rundresa, direktresa, ledare)

In [1]:
import sys
sys.path.insert(0, '/config/notebooks/wsj27')
import wsj27_utils as u
import pandas as pd
import numpy as np
import json
from collections import Counter

# Fetch all participants from Scoutnet
raw_data = u.fetch_participants()
df_all, skipped = u.build_participant_dataframe(raw_data)
u.assign_coordinates(df_all)

# Östergötland bounds
OG_BOUNDS = {'lat_min': 58.0, 'lat_max': 58.8, 'lng_min': 14.8, 'lng_max': 16.8}

# Filter to Östergötland by kår coordinates
df = df_all[
    (df_all['lat'] > OG_BOUNDS['lat_min']) & (df_all['lat'] < OG_BOUNDS['lat_max']) &
    (df_all['lng'] > OG_BOUNDS['lng_min']) & (df_all['lng'] < OG_BOUNDS['lng_max'])
].copy().reset_index(drop=True)

print(f'=== Östergötland ===')
print(f'Totalt: {len(df)} personer från {df["kar"].nunique()} kårer')

Fetched 1239 participants
Confirmed: 1180, Unconfirmed: 26, Cancelled: 33
Total confirmed participants: 1180
Skipped: 26 unconfirmed, 0 wrong age/no DOB

By category:
category
deltagare    989
ist          169
cmt           22

By travel type:
travel
rundresa      837
direktresa    202
egen_resa     119
other          22

Friend wishes:
  With friend 1 member no: 647
  With friend 2 member no: 366
  With friend 1 name (text): 57
  With friend 2 name (text): 38
With coordinates: 1173
Without coordinates (Sweden centroid): 7
=== Östergötland ===
Totalt: 61 personer från 19 kårer


In [2]:
# Övergripande statistik
print(f'=== Deltagare per kategori ===')
cat_counts = df['category'].value_counts()
for cat, n in cat_counts.items():
    print(f'  {cat:15s} {n:>4}')
print(f'  {"TOTALT":15s} {len(df):>4}')

print(f'\n=== Restyp ===')
travel_counts = df['travel'].value_counts()
for t, n in travel_counts.items():
    print(f'  {t:15s} {n:>4}')

print(f'\n=== Kön ===')
sex_counts = df['sex'].map(u.SEX_MAP).value_counts()
for s, n in sex_counts.items():
    print(f'  {s:15s} {n:>4}')

print(f'\n=== Åldersfördelning ===')
# Separate age stats for deltagare vs IST/CMT
df_del = df[df['category'] == 'deltagare']
df_ist = df[df['category'].isin(['ist', 'cmt'])]
print(f'\nDeltagare (14-17):')
for age in sorted(df_del['age'].unique()):
    n = len(df_del[df_del['age'] == age])
    bar = '█' * n
    print(f'  {age:>3}: {n:>3} {bar}')
if len(df_ist) > 0:
    print(f'\nIST/CMT:')
    ages = df_ist['age'].values
    print(f'  Antal: {len(df_ist)}, Ålder: {int(min(ages))}-{int(max(ages))}, Medel: {np.mean(ages):.0f}')

=== Deltagare per kategori ===
  deltagare         52
  ist                6
  cmt                3
  TOTALT            61

=== Restyp ===
  rundresa          35
  direktresa        17
  egen_resa          6
  other              3

=== Kön ===
  Man               32
  Kvinna            29

=== Åldersfördelning ===

Deltagare (14-17):
   14:  14 ██████████████
   15:  12 ████████████
   16:  12 ████████████
   17:  14 ██████████████

IST/CMT:
  Antal: 9, Ålder: 20-51, Medel: 32


In [3]:
# Statistik per kår
print(f'{"Kår":<45} {"Del":>4} {"IST":>4} {"CMT":>4} {"Tot":>4}  {"Rund":>4} {"Dir":>4} {"Egen":>4}  {"M":>3} {"K":>3}  Åldrar')
print('=' * 120)

totals = {'del': 0, 'ist': 0, 'cmt': 0, 'rund': 0, 'dir': 0, 'egen': 0, 'm': 0, 'k': 0}

for kar in sorted(df['kar'].unique()):
    k = df[df['kar'] == kar]
    n_del = len(k[k['category'] == 'deltagare'])
    n_ist = len(k[k['category'] == 'ist'])
    n_cmt = len(k[k['category'] == 'cmt'])
    n_rund = len(k[k['travel'] == 'rundresa'])
    n_dir = len(k[k['travel'] == 'direktresa'])
    n_egen = len(k[k['travel'] == 'egen_resa'])
    sex_c = k['sex'].map(u.SEX_MAP).value_counts()
    n_m = sex_c.get('Man', 0)
    n_k = sex_c.get('Kvinna', 0)
    ages = sorted(k[k['category']=='deltagare']['age'].unique())
    age_str = ','.join(str(int(a)) for a in ages) if ages else '-'
    
    totals['del'] += n_del; totals['ist'] += n_ist; totals['cmt'] += n_cmt
    totals['rund'] += n_rund; totals['dir'] += n_dir; totals['egen'] += n_egen
    totals['m'] += n_m; totals['k'] += n_k
    
    print(f'{kar:<45} {n_del:>4} {n_ist:>4} {n_cmt:>4} {len(k):>4}  {n_rund:>4} {n_dir:>4} {n_egen:>4}  {n_m:>3} {n_k:>3}  {age_str}')

print('=' * 120)
t = totals
print(f'{"TOTALT":<45} {t["del"]:>4} {t["ist"]:>4} {t["cmt"]:>4} {t["del"]+t["ist"]+t["cmt"]:>4}  {t["rund"]:>4} {t["dir"]:>4} {t["egen"]:>4}  {t["m"]:>3} {t["k"]:>3}')

Kår                                            Del  IST  CMT  Tot  Rund  Dir Egen    M   K  Åldrar
Berga Scoutkår                                   6    0    0    6     6    0    0    5   1  15,16,17
Equmenia Rimforsa                                2    0    0    2     0    2    0    1   1  15,17
Hanekinds scoutkår                               8    1    0    9     5    3    1    7   2  14,15,16,17
Johannelunds Scoutkår                            5    0    0    5     4    1    0    4   1  14,15,16,17
KFUM Norrköping Scout med medlemskap i KFUM Norrköping    0    0    1    1     0    0    0    1   0  -
Kvillinge scoutkår                               1    0    0    1     1    0    0    0   1  16
Kärna Scoutkår Linköping                         4    0    0    4     1    3    0    0   4  14,15
Landeryd Scoutkår                                3    0    2    5     3    0    0    2   3  14,15,16
Linghems Scoutkår                                3    2    0    5     3    0    2    3   2  14,15

In [4]:
# Alla deltagare per kår
for kar in sorted(df['kar'].unique()):
    k = df[df['kar'] == kar].sort_values(['category', 'name'])
    print(f'\n--- {kar} ({len(k)} st) ---')
    for _, p in k.iterrows():
        cat_str = p['category'].upper() if p['category'] != 'deltagare' else 'Del'
        sex_str = u.SEX_MAP.get(p['sex'], '?')
        travel_str = p['travel']
        print(f'  {p["name"]:<30} {cat_str:<4} {travel_str:<12} {int(p["age"]):>2} år  {sex_str}')


--- Berga Scoutkår (6 st) ---
  Elliot Widholm                 Del  rundresa     16 år  Man
  Franz Schmitterblad            Del  rundresa     15 år  Man
  Hugo Sterneberg                Del  rundresa     15 år  Man
  Katarina Severin               Del  rundresa     16 år  Kvinna
  Ludvig Dalbark                 Del  rundresa     15 år  Man
  Valter Dalbark                 Del  rundresa     17 år  Man

--- Equmenia Rimforsa (2 st) ---
  Alma Hultman                   Del  direktresa   15 år  Kvinna
  Vidar Hultman                  Del  direktresa   17 år  Man

--- Hanekinds scoutkår (9 st) ---
  Erik Vidbacken                 Del  direktresa   17 år  Man
  Ludvig Årstrand                Del  rundresa     17 år  Man
  Sam Johansson                  Del  rundresa     17 år  Kvinna
  Sebastian Stjärnström          Del  rundresa     15 år  Man
  Sofia Årstrand                 Del  rundresa     14 år  Kvinna
  Uno Jonasson                   Del  direktresa   17 år  Man
  Viktor Vidbacken  

In [5]:
# Gruppindelning - vilken grupp hamnade Östergötlands deltagare i?
# Ladda från befintliga gruppindelnings-CSV:er
import os

OUTPUT_DIR = '/config/notebooks/wsj27'
group_files = {
    'Rundresa': f'{OUTPUT_DIR}/wsj27_rundresa_grupper.csv',
    'Direktresa': f'{OUTPUT_DIR}/wsj27_direktresa_grupper.csv',
}

og_karer = set(df['kar'].unique())

for label, path in group_files.items():
    if not os.path.exists(path):
        print(f'{label}: {path} finns inte (kör gruppindelning först)')
        continue
    df_grp = pd.read_csv(path)
    df_og = df_grp[df_grp['kar'].isin(og_karer)].copy()
    if len(df_og) == 0:
        print(f'{label}: inga Östergötland-deltagare')
        continue
    
    print(f'\n=== {label}: {len(df_og)} personer från Östergötland ===')
    
    # Per group: how many from ÖG?
    grp_counts = df_og['group'].value_counts().sort_index()
    print(f'\nFördelning över grupper:')
    for grp, n in grp_counts.items():
        grp_total = len(df_grp[df_grp['group'] == grp])
        pct = n / grp_total * 100
        bar = '█' * n
        print(f'  Grupp {grp:>2}: {n:>2}/{grp_total} ({pct:>4.0f}%) {bar}')
    
    # Who is in which group?
    print(f'\nDeltagare per grupp:')
    for grp in sorted(df_og['group'].unique()):
        members = df_og[df_og['group'] == grp].sort_values('name')
        names = [f"{r['name']} ({r['kar'].split()[0]})" for _, r in members.iterrows()]
        print(f'  Grupp {grp:>2}: {", ".join(names)}')


=== Rundresa: 35 personer från Östergötland ===

Fördelning över grupper:
  Grupp  5:  1/36 (   3%) █
  Grupp 11:  1/36 (   3%) █
  Grupp 12:  2/36 (   6%) ██
  Grupp 13: 11/36 (  31%) ███████████
  Grupp 14: 15/36 (  42%) ███████████████
  Grupp 15:  4/36 (  11%) ████
  Grupp 22:  1/36 (   3%) █

Deltagare per grupp:
  Grupp  5: Elsa Thell (Vist)
  Grupp 11: Einar Selldin (Vreta)
  Grupp 12: Katarina Severin (Berga), Vilmer Däljemar (Vifolka)
  Grupp 13: Albert Lingborg (Svärtinge), Alma Brantvall (Åby), Fingal Gillholm (Johannelunds), Hedda Nordgren (Johannelunds), Hugo Sterneberg (Berga), Ida Lindqvist (Kvillinge), Josef Johansson (Johannelunds), Kajsa Berg (Kärna), Maja Larsson (Östra), Märta Strömberg (Landeryd), Otto Brantvall (Åby)
  Grupp 14: Amelie Karlsson (Landeryd), Dani Sam (Mjölby), Elliot Widholm (Berga), Franz Schmitterblad (Berga), Gunnar Wiqvist (Linghems), Ida Sandholdt (Landeryd), Ivar Frisell (Mjölby), Linus Darander (Linghems), Ludvig Dalbark (Berga), Ludvig Årst

In [6]:
# Ledare från Östergötland
ledare_path = f'{OUTPUT_DIR}/wsj27_ledare_grupper.csv'
if os.path.exists(ledare_path):
    df_led = pd.read_csv(ledare_path)
    # Match by bostadsort geocoding - check if any leader's kår is from ÖG
    df_led_og = df_led[df_led['kar'].isin(og_karer)]
    
    # Also check by lat/lng bounds for leaders
    if 'lat' in df_led.columns:
        df_led_geo = df_led[
            (df_led['lat'] > OG_BOUNDS['lat_min']) & (df_led['lat'] < OG_BOUNDS['lat_max']) &
            (df_led['lng'] > OG_BOUNDS['lng_min']) & (df_led['lng'] < OG_BOUNDS['lng_max'])
        ]
        # Combine both matches
        df_led_og = pd.concat([df_led_og, df_led_geo]).drop_duplicates()
    
    if len(df_led_og) > 0:
        print(f'=== Ledare från/i Östergötland: {len(df_led_og)} st ===')
        for _, r in df_led_og.sort_values('name').iterrows():
            bost = r.get('bostadsort', '?')
            print(f'  {r["name"]:<30} Grupp {int(r["group"]):>2}  {r["kar"]}  ({bost})')
    else:
        print('Inga ledare från Östergötland')
else:
    print(f'Ledare-CSV saknas: {ledare_path}')

Inga ledare från Östergötland


In [7]:
# Sammanfattning Östergötland
n_del = len(df[df['category'] == 'deltagare'])
n_ist = len(df[df['category'] == 'ist'])
n_cmt = len(df[df['category'] == 'cmt'])
n_rund = len(df[df['travel'] == 'rundresa'])
n_dir = len(df[df['travel'] == 'direktresa'])
sex_c = df['sex'].map(u.SEX_MAP).value_counts()

print(f'╔══════════════════════════════════════════╗')
print(f'║     WSJ 2027 - Östergötland              ║')
print(f'╠══════════════════════════════════════════╣')
print(f'║  Kårer:        {df["kar"].nunique():>4}                      ║')
print(f'║  Totalt:       {len(df):>4} personer               ║')
print(f'║                                          ║')
print(f'║  Deltagare:    {n_del:>4}                      ║')
print(f'║  IST:          {n_ist:>4}                      ║')
print(f'║  CMT:          {n_cmt:>4}                      ║')
print(f'║                                          ║')
print(f'║  Rundresa:     {n_rund:>4}                      ║')
print(f'║  Direktresa:   {n_dir:>4}                      ║')
print(f'║                                          ║')
print(f'║  Man:          {sex_c.get("Man", 0):>4}                      ║')
print(f'║  Kvinna:       {sex_c.get("Kvinna", 0):>4}                      ║')
print(f'╚══════════════════════════════════════════╝')

# Andel av hela kontingenten
print(f'\nAndel av hela kontingenten: {len(df)}/{len(df_all)} ({len(df)/len(df_all)*100:.1f}%)')

╔══════════════════════════════════════════╗
║     WSJ 2027 - Östergötland              ║
╠══════════════════════════════════════════╣
║  Kårer:          19                      ║
║  Totalt:         61 personer               ║
║                                          ║
║  Deltagare:      52                      ║
║  IST:             6                      ║
║  CMT:             3                      ║
║                                          ║
║  Rundresa:       35                      ║
║  Direktresa:     17                      ║
║                                          ║
║  Man:            32                      ║
║  Kvinna:         29                      ║
╚══════════════════════════════════════════╝

Andel av hela kontingenten: 61/1180 (5.2%)
